In [18]:
import numpy as np
import pandas as pd
from scipy.stats import linregress, pearsonr
from scipy.special import comb

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

In [19]:
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all independent variables
independent_variables = np.concatenate((np.array(species), np.array(controls)))

In [26]:
# predictions for all possible conditions
pred_df = pd.read_csv("space/space.csv")

In [28]:
pred_df

,Experiments,Time,ACabs,BAabs,BHabs,BLabs,BUabs,CAabs,CCabs,CHabs,...,Lactate,Butyrate,Acetate,AcGum,ArGal,Inulin,Pectin,Starch,Xylan,Objective
0,RI,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
1,RI,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.042092,1.906625,14.540738,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
2,RI,2.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,NaN,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
3,RI,3.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.669601,1.656765,6.417598,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.049654
4,RI-Xylan,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.335008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8388347,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,3.0,0.022939,0.193776,0.000000,0.000000,0.182286,0.000000,0.136966,0.000000,...,0.240772,8.001605,22.746085,0.200000,0.200000,0.200000,0.200000,0.200000,0.000000,0.858180
8388348,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,0.0,0.000667,0.000667,0.000667,0.000667,0.000667,0.000667,0.000667,0.000667,...,0.000000,0.000000,0.000000,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.910289
8388349,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,1.0,0.034305,0.118248,0.006939,0.016604,0.309078,0.000000,0.067389,0.013777,...,0.681623,10.654917,16.179780,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.910289
8388350,AC-BA-BH-BL-BU-CA-CC-CH-DF-EL-ER-FP-PC-PJ-RI-A...,2.0,0.025429,0.158405,0.000000,0.000000,0.283465,0.000000,0.190207,0.000000,...,NaN,NaN,NaN,0.166667,0.166667,0.166667,0.166667,0.166667,0.166667,0.910289


In [31]:
# simulated data
E = np.array(pred_df.iloc[pred_df.Time.values==0]['Experiments'].values)
X = np.array(pred_df.iloc[pred_df.Time.values==0][species+controls].values) 
y = pred_df.iloc[pred_df.Time.values==3]['Butyrate'].values

# make X binary
X = np.array(X>0, int)

In [32]:
def find_row_index(X, columns):
    """
    Returns the index of the row in X that exactly matches the vector x.
    
    Parameters:
    X (numpy.ndarray): The input n x m matrix.
    x (numpy.ndarray): The m-dimensional vector to match.
    
    Returns:
    int: The index of the matching row, or -1 if no match is found.
    """
    # define vector x that matches motif
    x = np.zeros(X.shape[-1])
    x[list(columns)] = 1.
    
    # Check for rows that match the vector x element-wise
    match = np.all(X == x, axis=1)
    
    # If a match is found, return the index; otherwise, return -1
    indices = np.where(match)[0]
    return indices[0] if indices.size > 0 else -1

In [43]:
# save best motifs in a dictionary
motif_dict = {}

# Value of k
for k in [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]:

    # Loop over all n choose k subsets
    motifs = []
    max_vals = []
    total = comb(len(independent_variables), k) 
    for motif in tqdm(itertools.combinations(range(X.shape[-1]), k), total=total):
        
        # save motif name
        motifs.append(motif)
        
        # index of all samples that satisfy motif
        motif_idx = find_row_index(X, motif)    
        
        # save minimum predicted value
        if motif_idx > -1:
            max_vals.append(y[motif_idx])
        
    # save best motif
    motif_observed = independent_variables[list(motifs[np.argmax(max_vals)])]
    
    # convert motif to hashable string
    motif_name = ""
    for m in motif_observed:
        motif_name += m.split('abs')[0] + '-'
    motif_name = motif_name[:-1]
    
    # save to dict
    motif_dict[motif_name] = np.max(max_vals)

100%|█████████████████████████████████████████| 21/21.0 [00:00<00:00, 27.93it/s]
100%|███████████████████████████████████████| 210/210.0 [00:07<00:00, 29.19it/s]


In [42]:
motif_dict

{'AC-Xylan': 12.26444037482702,
 'AC-BU-Inulin': 15.848078003437722,
 'AC-BU-CH-Inulin': 17.189964397964843,
 'AC-BU-CH-RI-Inulin': 18.00451078420552,
 'AC-BL-BU-ER-RI-Xylan': 18.85695083327783,
 'AC-BH-BL-BU-ER-RI-Xylan': 19.74976760159866,
 'AC-BH-BL-BU-ER-RI-Starch-Xylan': 20.625204417799864,
 'AC-BH-BL-BU-DF-ER-RI-Starch-Xylan': 20.85818898099338,
 'AC-BH-BL-BU-DF-ER-FP-PJ-RI-Xylan': 20.3694164224919,
 'AC': 2.435650382789245}

In [45]:
df = pd.DataFrame()
df['motifs'] = motif_dict.keys()
df['butyrate'] = motif_dict.values()
df

,motifs,butyrate
0,AC-Xylan,12.264440
1,AC-BU-Inulin,15.848078
2,AC-BU-CH-Inulin,17.189964
3,AC-BU-CH-RI-Inulin,18.004511
4,AC-BL-BU-ER-RI-Xylan,18.856951
5,AC-BH-BL-BU-ER-RI-Xylan,19.749768
6,AC-BH-BL-BU-ER-RI-Starch-Xylan,20.625204
7,AC-BH-BL-BU-DF-ER-RI-Starch-Xylan,20.858189
8,AC-BH-BL-BU-DF-ER-FP-PJ-RI-Xylan,20.369416
9,AC,2.435650


In [46]:
df.to_csv("insights/mini_max_motifs.csv", index=False)